## Initial Data Load & Preview
The dataset is loaded from the raw data directory and inspected to understand its basic structure and contents.

Each row in the dataset represents a **single financial transaction** associated with an account. Transactions include both merchant payments and peer-to-peer (P2P) transfers.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/simulated_fake_transactions_dataset_2.csv")

df.head()

,Date,Timestamp,Account No,Balance,Amount,Third Party Account No,Third Party Name
0,01/01/2025,00:00,839964633.0,3385.0,2602.0,NaN,Boots
1,01/01/2025,00:00,793069740.0,3318.0,2780.0,NaN,Mountain Warehouse
2,01/01/2025,00:00,793069740.0,2278.0,-1040.0,705519819.0,NaN
3,01/01/2025,00:00,705519819.0,2726.0,1040.0,793069740.0,NaN
4,01/01/2025,00:00,726302171.0,3253.5,1542.5,NaN,Craftastic


### Key Observations
The dataset contains transaction-level records with:
- Transaction date and time
- Account identifier
- Transaction amount
- Account balance after transaction
- Merchant information or third-party account details

Transaction amounts include both **positive and negative values**
* Positive amounts represent inflows
* Negative amounts represent outflows

Peer-to-Peer transactions can be identified when a `Third Party Account No` is present and 'Third Party Name' is missing.

## Data Structure and Summary Statistics
This section examines the dataset size, schema, data types, and high-level statistics to assess data quality and analytical readiness.

In [5]:
df.shape
df.info()
df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287791 entries, 0 to 287790
Data columns (total 7 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Date                    287510 non-null  object 
 1   Timestamp               287524 non-null  object 
 2   Account No              287480 non-null  float64
 3   Balance                 287506 non-null  float64
 4   Amount                  287482 non-null  float64
 5   Third Party Account No  8681 non-null    float64
 6   Third Party Name        278821 non-null  object 
dtypes: float64(4), object(3)
memory usage: 15.4+ MB


,Date,Timestamp,Account No,Balance,Amount,Third Party Account No,Third Party Name
count,287510,287524,2.874800e+05,287506.000000,287482.000000,8.681000e+03,278821
unique,344,1412,NaN,NaN,NaN,NaN,94
top,01/01/2025,23:59,NaN,NaN,NaN,NaN,Coop Local
freq,6975,56259,NaN,NaN,NaN,NaN,39292
mean,NaN,NaN,5.466498e+08,2411.670723,1.008623,5.385943e+08,NaN
std,NaN,NaN,2.557615e+08,2938.549210,454.844927,2.582319e+08,NaN
min,NaN,NaN,1.001007e+08,-4716.841031,-9310.000000,1.001007e+08,NaN
25%,NaN,NaN,3.302819e+08,342.182166,-74.020000,3.083120e+08,NaN
50%,NaN,NaN,5.396997e+08,1551.429788,-28.280000,5.332414e+08,NaN
75%,NaN,NaN,7.676272e+08,3956.245277,-11.560000,7.675980e+08,NaN


### Dataset Size
* **Total rows:** 287,791 transactions
* **Total columns:** 7

### Column Overview
| Column Name | Description |
|-------------|-------------|
| Date | Transaction date |
| Timestamp | Transaction time |
| Account No | Unique identifier for a customer account |
| Balance | Account balance after transaction |
| Amount | Transaction amount (positive = inflow, negative = outflow) |
| Third Party Account No | Counterparty account for P2P transfers |
| Third Party Name | Merchant name or counterparty label |

### Granularity
* The dataset is at **transaction-level granularity**
* Multiple transactions can occur per account per day

### Data Type Observations
- Date and time fields are currently stored as **object (string)** types and will require conversion
- Account identifiers are numeric but should be treates as **categorical IDs**
- Monetary fields ('Amount', 'Balance') are stored as floats

### Statistical Insights
* Transaction amounts range from approximately **-9,310 to +13,040**
* Account balances vary widely, indiccating diverse financial behaviors
* The majority of transactions are associated with merchants, while P2P transfer represent a smaller subset

## Date & Amount Sanity Checks

In [8]:
df["transaction_datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Timestamp"],
    errors="coerce"
)

df[["Date", "Timestamp", "transaction_datetime"]].head()

,Date,Timestamp,transaction_datetime
0,01/01/2025,00:00,2025-01-01
1,01/01/2025,00:00,2025-01-01
2,01/01/2025,00:00,2025-01-01
3,01/01/2025,00:00,2025-01-01
4,01/01/2025,00:00,2025-01-01


In [9]:
df["transaction_datetime"].min(), df["transaction_datetime"].max()

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-12-11 21:21:00'))

In [10]:
df["Amount"].min(), df["Amount"].max()
df["Balance"].min(), df["Balance"].max()

(-4716.841031358929, 14665.944386813751)

In [11]:
df[df["Amount"] == 0].shape
df[df["Balance"] < 0].shape

(48688, 8)

This step validates the temporal coverage of the dataset and checks for anomalies in transaction amounts and account balances.

### Date Range Validation
* Transactions span from **January 1, 2025** to **December 11, 2025**
* The dataset covers nearly a full calenday year, making it suitable for:
    * Monthly aggregation
    * Trend analysis
    * Financial health scoring
 
No future-dated transactions were observed.

### Transaction Amount Validation
* Transaction amounts range from approximately **-4,717 to +14,666**
* Both positive and negative values are present:
    * Positive amounts indicate inflows
    * Negative amounts indicate outflows
 
Zero-amount transactions are present and will be reviewd during data quality checks.

### Balance Validation
* Account balances include negative values
* Negative balances are considered valid and may represent:
    * Overdraft usage
    * Short-term liquidity stress

In [13]:
# ---- FIXING IDENTIFIER TYPES ----
df['Account No'] = df['Account No'].astype('Int64').astype('string')
df['Third Party Account No'] = df['Third Party Account No'].astype('Int64').astype('string')

## Missing Values & Data Quality Classification

In [15]:
df.isna().sum().sort_values(ascending=False)

Third Party Account No    279110
transaction_datetime      178299
Third Party Name            8970
Account No                   311
Amount                       309
Balance                      285
Date                         281
Timestamp                    267
dtype: int64

### Structural Missing Values (Expected)
* `Third Party Account No`:
    * Missing for most transactions
    * Indicates non-P2P transactions (merchant payments)
* `Third Party Name`:
    * Missing primarily for peer-to-peer transfers
    * Expected behavior and retained as-is
 
### Derived Missing Values
* `transaction_datetime`:
    * Missing when either `Date` or `Timestamp` is missing
    * This is a consequence of datetime construction rather than a standalone data issue
 
### True Data Quality Issues
A small number of records contain missing values in critical analytical fields:
* `Account No`
* `Amount`
* `Balance`
* `Date` or `Timestamp`

These records cannot be reliably attributed to an account or time period and will be excluded from KPI aggregation.

### Cleaning Strategy
* Preserve all structurally valid transaction records
* Exclude transactions missing:
    * Account identifier
    * Transaction amount
    * Account balance
    * Transaction datetime
* Retain missing values that encode transaction type (e.g., P2P vs merchant)

In [17]:
df_clean = df.dropna(
    subset=["Account No", "Amount", "Balance", "transaction_datetime"]
).copy()

df_clean.shape

(109160, 8)

In [18]:
df_clean.to_csv("../data/processed/transactions_clean.csv", index=False)